# Trust Drift PoC

In [37]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
monday_data = pd.read_csv('trust_drift/data/raw/Monday-WorkingHours.pcap_ISCX.csv')

print("=== BASIC INFO ===")
print(f"Rows:    {len(monday_data):,}")
print(f"Columns: {len(monday_data.columns)}")
print(f"\nData Types:")
print(monday_data.dtypes.value_counts())

=== BASIC INFO ===
Rows:    529,918
Columns: 79

Data Types:
int64      54
float64    24
object      1
Name: count, dtype: int64


In [39]:
print("=== NULL VALUES ===")
null_counts = monday_data.isnull().sum()
null_cols = null_counts[null_counts > 0]

if len(null_cols) == 0:
    print("No null values found!")
else:
    print(f"Columns with nulls:")
    print(null_cols)
    print(f"\nTotal null values: {null_counts.sum()}")

=== NULL VALUES ===
Columns with nulls:
Flow Bytes/s    64
dtype: int64

Total null values: 64


In [40]:
print("=== INFINITY VALUES ===")
numeric_cols = monday_data.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(monday_data[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

if len(inf_cols) == 0:
    print("No infinity values found!")
else:
    print(f"Columns with infinity:")
    print(inf_cols)
    print(f"\nTotal infinity values: {inf_counts.sum()}")

=== INFINITY VALUES ===
Columns with infinity:
Flow Bytes/s       373
 Flow Packets/s    437
dtype: int64

Total infinity values: 810


In [41]:
print("=== LABELS ===")
monday_data.columns = monday_data.columns.str.strip()
print(monday_data['Label'].value_counts())

=== LABELS ===
Label
BENIGN    529918
Name: count, dtype: int64


In [42]:
print("=== CLEANING DATA ===")

# Step 1 - Strip column name spaces
monday_data.columns = monday_data.columns.str.strip()

# Step 2 - Replace infinity with NaN
monday_data = monday_data.replace([np.inf, -np.inf], np.nan)

# Step 3 - Drop all NaN rows
before = len(monday_data)
monday_data = monday_data.dropna()
after = len(monday_data)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}")
print(f"Rows removed:         {before - after:,}")
print(f"Labels after:         {monday_data['Label'].value_counts().to_dict()}")
print("\nMonday data cleaned!")

=== CLEANING DATA ===
Rows before cleaning: 529,918
Rows after cleaning:  529,481
Rows removed:         437
Labels after:         {'BENIGN': 529481}

Monday data cleaned!


In [43]:

tuesday_data = pd.read_csv('trust_drift/data/raw/Tuesday-WorkingHours.pcap_ISCX.csv')

# Clean
tuesday_data.columns = tuesday_data.columns.str.strip()
tuesday_data = tuesday_data.replace([np.inf, -np.inf], np.nan)
before = len(tuesday_data)
tuesday_data = tuesday_data.dropna()
after = len(tuesday_data)

print(f"Tuesday:")
print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Removed:     {before - after:,}")
print(f"Labels:      {tuesday_data['Label'].value_counts().to_dict()}")

Tuesday:
Rows before: 445,909
Rows after:  445,645
Removed:     264
Labels:      {'BENIGN': 431813, 'FTP-Patator': 7935, 'SSH-Patator': 5897}


In [53]:
import pandas as pd
import numpy as np
import os

# ── Correct Paths ──────────────────────────────────
raw_path     = r'd:\Arfat\CyberCell\Trust_Drift\Trust_Drift_PoC\trust_drift\data\raw'
cleaned_path = r'd:\Arfat\CyberCell\Trust_Drift\Trust_Drift_PoC\trust_drift\data\cleaned'

# ── Cleaning Function ──────────────────────────────
def clean_data(filepath, name):
    print(f"\nLoading {name}...")
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df = df.replace([np.inf, -np.inf], np.nan)
    before = len(df)
    df = df.dropna()
    after = len(df)
    print(f"Rows before:  {before:,}")
    print(f"Rows after:   {after:,}")
    print(f"Rows removed: {before - after:,}")
    print(f"Labels:       {df['Label'].value_counts().to_dict()}")
    return df

# ── Load and Clean ─────────────────────────────────
monday_data    = clean_data(os.path.join(raw_path, 'Monday-WorkingHours.pcap_ISCX.csv'),    'Monday')
tuesday_data   = clean_data(os.path.join(raw_path, 'Tuesday-WorkingHours.pcap_ISCX.csv'),   'Tuesday')
wednesday_data = clean_data(os.path.join(raw_path, 'Wednesday-workingHours.pcap_ISCX.csv'), 'Wednesday')

# ── Save ───────────────────────────────────────────
print("\nSaving cleaned files...")
monday_data.to_csv(os.path.join(cleaned_path, 'monday_clean.csv'),       index=False)
tuesday_data.to_csv(os.path.join(cleaned_path, 'tuesday_clean.csv'),     index=False)
wednesday_data.to_csv(os.path.join(cleaned_path, 'wednesday_clean.csv'), index=False)

# ── Verify ─────────────────────────────────────────
print("\nFiles saved:")
for f in os.listdir(cleaned_path):
    size = os.path.getsize(os.path.join(cleaned_path, f)) / (1024*1024)
    print(f"  {f}  →  {size:.1f} MB")

print("\nData cleaning complete! ✅")


Loading Monday...
Rows before:  529,918
Rows after:   529,481
Rows removed: 437
Labels:       {'BENIGN': 529481}

Loading Tuesday...
Rows before:  445,909
Rows after:   445,645
Rows removed: 264
Labels:       {'BENIGN': 431813, 'FTP-Patator': 7935, 'SSH-Patator': 5897}

Loading Wednesday...
Rows before:  692,703
Rows after:   691,406
Rows removed: 1,297
Labels:       {'BENIGN': 439683, 'DoS Hulk': 230124, 'DoS GoldenEye': 10293, 'DoS slowloris': 5796, 'DoS Slowhttptest': 5499, 'Heartbleed': 11}

Saving cleaned files...

Files saved:
  monday_clean.csv  →  182.9 MB
  tuesday_clean.csv  →  141.1 MB
  wednesday_clean.csv  →  232.8 MB

Data cleaning complete! ✅
